In [1]:
import torch, transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)

from transformers import pipeline
gen = pipeline("text-generation", model="gpt2")
print(gen("The future of AI is", max_new_tokens=20)[0]["generated_text"])

torch: 2.15.0+cpu | transformers: 5.22.2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

d:\captain garuda\roadmap\llm-agentic-ai-engineering\project\math-for-llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NITRO 5\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\captain garuda\roadmap\llm-agentic-ai-engineering\project\math-for-llm\.venv\Lib

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

... (2 outputs left out)

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

The future of AI is now clearer than ever.

Read next: It's all about finding the best ways to give


## 2. Tokenizer: from text to numbers

In [5]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
print("Vocab size:", tok.vocab_size)

text = "The cat sat on the mat."
ids = tok.encode(text)
tokens = tok.convert_ids_to_tokens(ids)

for i, t in zip(ids, tokens):
  print(f"{i:>6} {t!r}")

print("\nDecoding:", tok.decode(ids))

Vocab size: 50257
   464 'The'
  3797 'Ġcat'
  3332 'Ġsat'
   319 'Ġon'
   262 'Ġthe'
  2603 'Ġmat'
    13 '.'

Decoding: The cat sat on the mat.


In [4]:
samples = {
    "English":   "Artificial intelligence is changing the way we work.",
    "Indonesia": "Kecerdasan buatan sedang mengubah cara kita bekerja.",
    "Code":      "def add(a, b): return a + b",
    "Numbers":   "1234567 + 89 = 1234656",
}

for name, s in samples.items():
    toks = tok.tokenize(s)
    print(f"{name:10} {len(s.split()):>2} words -> {len(toks):>2} tokens")
    print("           ", toks, "\n")


English     8 words -> 10 tokens
            ['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġchanging', 'Ġthe', 'Ġway', 'Ġwe', 'Ġwork', '.'] 

Indonesia   7 words -> 21 tokens
            ['K', 'ec', 'erd', 'as', 'an', 'Ġbu', 'atan', 'Ġsed', 'ang', 'Ġmen', 'g', 'ub', 'ah', 'Ġcar', 'a', 'Ġk', 'ita', 'Ġbe', 'ker', 'ja', '.'] 

Code        7 words -> 11 tokens
            ['def', 'Ġadd', '(', 'a', ',', 'Ġb', '):', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb'] 

Numbers     5 words ->  9 tokens
            ['123', '45', '67', 'Ġ+', 'Ġ89', 'Ġ=', 'Ġ12', '34', '656'] 



**Findings**
- English: 1.25 tokens/word. Indonesian: 3.0 tokens/word, so 2.4x more tokens for the same meaning.
- "Kecerdasan" becomes 5 meaningless pieces. BPE merges by frequency, not by morphology.
- 1234567 and 1234656 are split in different patterns (123|45|67 vs 12|34|656), so the model never sees a consistent place value.
- Root cause for both: GPT-2 saw almost no Indonesian text during training.

## 3. Inside the model: logits, softmax, next token

In [6]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

prompt = "The cat sat on the"
inputs = tok(prompt, return_tensors="pt")
print("input_ids:", inputs["input_ids"])

with torch.no_grad():
    out = model(**inputs)

print("logits shape:", out.logits.shape)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

input_ids: tensor([[ 464, 3797, 3332,  319,  262]])
logits shape: torch.Size([1, 5, 50257])


In [7]:
last_logits = out.logits[0, -1]            # scores for the token after "the"
probs = torch.softmax(last_logits, dim=-1)

top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"{p.item():7.2%}  {tok.decode(i)!r}")

print("\nSum of all probabilities:", probs.sum().item())


  7.64%  ' floor'
  6.53%  ' bed'
  5.41%  ' couch'
  5.21%  ' ground'
  4.78%  ' edge'
  3.16%  ' bench'
  3.15%  ' table'
  2.88%  ' sofa'
  1.99%  ' other'
  1.90%  ' back'

Sum of all probabilities: 1.0000098943710327


**Findings**
- Logits shape (1, 5, 50257): one score per vocab token at every position. Only the last position is used to generate.
- The top token " floor" has only 7.64%. The model is uncertain and the probability is spread over many valid options.
- Greedy decoding is deterministic and discards the other 92%. The output starts to repeat ("the cat", empty lines).

In [9]:
ids = tok("The cat sat on the", return_tensors="pt")["input_ids"]

for step in range(15):
    with torch.no_grad():
        next_logits = model(ids).logits[0, -1]
    next_id = next_logits.argmax()                      # always pick the highest
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)   # append, then repeat
    print(f"{step:>2}: {tok.decode(ids[0])}")


 0: The cat sat on the floor
 1: The cat sat on the floor,
 2: The cat sat on the floor, and
 3: The cat sat on the floor, and the
 4: The cat sat on the floor, and the cat
 5: The cat sat on the floor, and the cat was
 6: The cat sat on the floor, and the cat was still
 7: The cat sat on the floor, and the cat was still asleep
 8: The cat sat on the floor, and the cat was still asleep.
 9: The cat sat on the floor, and the cat was still asleep.

10: The cat sat on the floor, and the cat was still asleep.


11: The cat sat on the floor, and the cat was still asleep.

"
12: The cat sat on the floor, and the cat was still asleep.

"I
13: The cat sat on the floor, and the cat was still asleep.

"I'm
14: The cat sat on the floor, and the cat was still asleep.

"I'm sorry
